# SpecTNT — Colab Training Pipeline

Trains all 3 model configs (HarmonicCNN, SpecTNT, SpecTNT+CTL) across 4 folds.
Results are saved back to Google Drive.

In [ ]:
# Mount Google Drive where harmonix data lives
from google.colab import drive
drive.mount('/content/drive')

# ── CONFIG ──
GITHUB_REPO = "https://github.com/fharookshaik/BTU_RM_So26.git"
DRIVE_DATA  = "/content/drive/MyDrive/harmonixset"  # upload data/harmonixset here
DRIVE_OUT   = "/content/drive/MyDrive/spectnt_outputs"  # results land here
# ────────────

In [ ]:
import os, sys
os.chdir('/content')

# Clone repo
if not os.path.isdir('BTU_RM_So26'):
    !git clone {GITHUB_REPO}
os.chdir('BTU_RM_So26')

# Install deps
!pip install -q . 2>&1 | tail -5
!pip install -q torchcodec 2>&1 | tail -3  # optional, may not be needed

# Symlink data
os.makedirs('data', exist_ok=True)
if not os.path.islink('data/harmonixset'):
    !ln -s "{DRIVE_DATA}" data/harmonixset
    print(f"Symlinked {DRIVE_DATA} -> data/harmonixset")

In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device count  : {torch.cuda.device_count()}")
print(f"Device name   : {torch.cuda.get_device_name(0)}")
print(f"Torch version : {torch.__version__}")

In [ ]:
# ── Run all 12 experiments ──
import subprocess, sys, time

models = ["harmonic_cnn", "spectnt", "spectnt_ctl"]
folds  = list(range(4))

results_dir = "/content/spectnt_outputs"
os.makedirs(results_dir, exist_ok=True)

for model in models:
    for fold in folds:
        print(f"\n{'='*60}\nStarting: {model}, Fold {fold}\n{'='*60}")
        cmd = [sys.executable, "main.py", "--model", model, "--fold", str(fold)]
        t0 = time.time()
        result = subprocess.run(cmd, capture_output=True, text=True)
        elapsed = time.time() - t0
        print(result.stdout[-1000:] if len(result.stdout) > 1000 else result.stdout)
        if result.returncode != 0:
            print(f"ERROR (rc={result.returncode}):", result.stderr[-500:])
            print(f"Saving partial results and continuing...")
        print(f"Completed in {elapsed:.1f}s")

In [ ]:
# Copy results back to Drive
!mkdir -p "{DRIVE_OUT}"
!cp -r outputs/ "{DRIVE_OUT}/outputs/"
print(f"Results copied to {DRIVE_OUT}/outputs/")
print("Done!")